In [82]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

### Load Dataset FD001

In [83]:
# Nama kolom
index_names = ['unit_number', 'time_cycles']
operational_settings = ['setting_1', 'setting_2', 'setting_3']
sensor_names = [
    'T2', 'T24', 'T30', 'T50', 'P2', 'P15', 'P30',        # Temperatures & Pressures
    'Nf', 'Nc', 'epr', 'Ps30', 'phi', 'NRf', 'NRc',       # Rotational Speeds
    'BPR', 'farB', 'htBleed', 'Nf_dmd', 'PCNfR_dmd', 'W31', 'W32' # Others
]
columns = index_names + operational_settings + sensor_names

# Load train & test
df_train = pd.read_csv('../data/train_FD001.txt', sep='\s+', header=None, names=columns)
df_test  = pd.read_csv('../data/test_FD001.txt',  sep='\s+', header=None, names=columns)
df_rul   = pd.read_csv('../data/RUL_FD001.txt',   sep='\s+', header=None, names=['RUL'])

print(f'Train shape : {df_train.shape}')
print(f'Test shape  : {df_test.shape}')
print(f'RUL shape   : {df_rul.shape}')
df_train.head()

Train shape : (20631, 26)
Test shape  : (13096, 26)
RUL shape   : (100, 1)


,unit_number,time_cycles,setting_1,setting_2,setting_3,T2,T24,T30,T50,P2,P15,P30,Nf,Nc,epr,Ps30,phi,NRf,NRc,BPR,farB,htBleed,Nf_dmd,PCNfR_dmd,W31,W32
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.00,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [84]:
print(f"null train : {df_train.isnull().sum().sum()}")
print(f"null test : {df_test.isnull().sum().sum()}")

null train : 0
null test : 0


### drop constant columns

In [85]:
constant_columns = df_train.columns[df_train.nunique() == 1]
constant_columns

Index(['setting_3', 'T2', 'P2', 'epr', 'farB', 'Nf_dmd', 'PCNfR_dmd'], dtype='object')

In [86]:
df_train.drop(columns=constant_columns, axis=1, inplace=True)
df_test.drop(columns=constant_columns, axis=1, inplace=True)

In [87]:
df_train.head()

,unit_number,time_cycles,setting_1,setting_2,T24,T30,T50,P15,P30,Nf,Nc,Ps30,phi,NRf,NRc,BPR,htBleed,W31,W32
0,1,1,-0.0007,-0.0004,641.82,1589.70,1400.60,21.61,554.36,2388.06,9046.19,47.47,521.66,2388.02,8138.62,8.4195,392,39.06,23.4190
1,1,2,0.0019,-0.0003,642.15,1591.82,1403.14,21.61,553.75,2388.04,9044.07,47.49,522.28,2388.07,8131.49,8.4318,392,39.00,23.4236
2,1,3,-0.0043,0.0003,642.35,1587.99,1404.20,21.61,554.26,2388.08,9052.94,47.27,522.42,2388.03,8133.23,8.4178,390,38.95,23.3442
3,1,4,0.0007,0.0000,642.35,1582.79,1401.87,21.61,554.45,2388.11,9049.48,47.13,522.86,2388.08,8133.83,8.3682,392,38.88,23.3739
4,1,5,-0.0019,-0.0002,642.37,1582.85,1406.22,21.61,554.00,2388.06,9055.15,47.28,522.19,2388.04,8133.80,8.4294,393,38.90,23.4044


### Labeling RUL (Remaining Useful Life)

In [88]:
# RUL Calculation
def calculate_rul(data, unit_col='unit_number', cycle_col='time_cycles'):
    max_cycle = (
        data.groupby(unit_col)[cycle_col].max()
        .reset_index().rename(columns={cycle_col: 'max_of_unit'}))
    data = data.merge(max_cycle, on=unit_col, how='left')
    data['RUL'] = data['max_of_unit'] - data[cycle_col]
    data.drop('max_of_unit', axis=1, inplace=True)
    return data


df_train = calculate_rul(df_train)
df_train[['unit_number', 'time_cycles', 'RUL']].head()

,unit_number,time_cycles,RUL
0,1,1,191
1,1,2,190
2,1,3,189
3,1,4,188
4,1,5,187


In [89]:
# RUL Clipping
RUL_THRESHOLD = 125
df_train['RUL'] = df_train['RUL'].clip(upper=RUL_THRESHOLD)
print(f"RUL Clipping applied. Max RUL is now capped at {RUL_THRESHOLD} cycles.")
print("\nRUL Statistics after Clipping:")

df_train['RUL'].describe().T

RUL Clipping applied. Max RUL is now capped at 125 cycles.

RUL Statistics after Clipping:


count    20631.000000
mean        86.829286
std         41.673699
min          0.000000
25%         51.000000
50%        103.000000
75%        125.000000
max        125.000000
Name: RUL, dtype: float64

### Normalisasi (MinMaxScaler)

In [90]:
features = [c for c in df_train.columns if c not in ['unit_number', 'time_cycles', 'RUL']]
scaler = MinMaxScaler()
df_train[features] = scaler.fit_transform(df_train[features])
df_test[features] = scaler.transform(df_test[features])

### Simpan Dataset

In [91]:
df_train.head()

,unit_number,time_cycles,setting_1,setting_2,T24,T30,T50,P15,P30,Nf,Nc,Ps30,phi,NRf,NRc,BPR,htBleed,W31,W32,RUL
0,1,1,0.459770,0.166667,0.183735,0.406802,0.309757,1.0,0.726248,0.242424,0.109755,0.369048,0.633262,0.205882,0.199608,0.363986,0.333333,0.713178,0.724662,125
1,1,2,0.609195,0.250000,0.283133,0.453019,0.352633,1.0,0.628019,0.212121,0.100242,0.380952,0.765458,0.279412,0.162813,0.411312,0.333333,0.666667,0.731014,125
2,1,3,0.252874,0.750000,0.343373,0.369523,0.370527,1.0,0.710145,0.272727,0.140043,0.250000,0.795309,0.220588,0.171793,0.357445,0.166667,0.627907,0.621375,125
3,1,4,0.540230,0.500000,0.343373,0.256159,0.331195,1.0,0.740741,0.318182,0.124518,0.166667,0.889126,0.294118,0.174889,0.166603,0.333333,0.573643,0.662386,125
4,1,5,0.390805,0.333333,0.349398,0.257467,0.404625,1.0,0.668277,0.242424,0.149960,0.255952,0.746269,0.235294,0.174734,0.402078,0.416667,0.589147,0.704502,125


In [92]:
# ==========================================
# PERSIAPAN DATA XGBOOST (2D Tabular)
# ==========================================
# Train: Simpan semua baris
df_train.to_csv(os.path.join('../processed/', 'train_2d.csv'), index=False)

df_test.to_csv(os.path.join('../processed/', 'test_full.csv'), index=False)

# Test: Aturan C-MAPSS, evaluasi hanya pada siklus paling akhir dari tiap mesin
df_test_last = df_test.groupby('unit_number').last().reset_index()

# Gabungkan dengan True RUL (Label asli dari file RUL_FD001.txt)
df_test_last['RUL_True'] = df_rul['RUL']
df_test_last.to_csv(os.path.join('../processed/', 'test_2d.csv'), index=False)

# ==========================================
# PERSIAPAN DATA LSTM (3D Sequence)
# ==========================================
sequence_length = 30 

def gen_sequence(id_df, seq_length, seq_cols):
    data_array = id_df[seq_cols].values
    num_elements = data_array.shape[0]
    for start, stop in zip(range(0, num_elements-seq_length), range(seq_length, num_elements)):
        yield data_array[start:stop, :]

# A. Pembuatan Sequence untuk Train (Semua sequence yang mungkin)
seq_gen_train = (list(gen_sequence(df_train[df_train['unit_number']==id], sequence_length, features)) 
                 for id in df_train['unit_number'].unique())
seq_array_train = np.concatenate(list(seq_gen_train)).astype(np.float32)

# Label RUL untuk Train (Hanya mengambil target pada akhir setiap sequence)
def gen_labels(id_df, seq_length, label):
    data_array = id_df[label].values
    num_elements = data_array.shape[0]
    return data_array[seq_length:num_elements, :]

label_gen_train = [gen_labels(df_train[df_train['unit_number']==id], sequence_length, ['RUL']) 
                   for id in df_train['unit_number'].unique()]
label_array_train = np.concatenate(label_gen_train).astype(np.float32)

# B. Pembuatan Sequence untuk Test (HANYA sequence paling terakhir dari setiap unit)
seq_array_test_last = [df_test[df_test['unit_number']==id][features].values[-sequence_length:] 
                       for id in df_test['unit_number'].unique()]
seq_array_test_last = np.asarray(seq_array_test_last).astype(np.float32)

# Label RUL untuk Test (Ambil langsung dari file df_rul)
label_array_test = df_rul['RUL'].values.astype(np.float32)

# ==========================================
# SIMPAN SEMUA DATA 3D UNTUK LSTM
# ==========================================
np.save(os.path.join('../processed/', 'train_3d_features.npy'), seq_array_train)
np.save(os.path.join('../processed/', 'train_3d_labels.npy'), label_array_train)
np.save(os.path.join('../processed/', 'test_3d_features.npy'), seq_array_test_last)
np.save(os.path.join('../processed/', 'test_3d_labels.npy'), label_array_test)

print(f"Selesai! Data 2D disimpan.")
print(f"Shape 3D Train Features: {seq_array_train.shape}, Labels: {label_array_train.shape}")
print(f"Shape 3D Test Features : {seq_array_test_last.shape}, Labels: {label_array_test.shape}")

Selesai! Data 2D disimpan.
Shape 3D Train Features: (17631, 30, 17), Labels: (17631, 1)
Shape 3D Test Features : (100, 30, 17), Labels: (100,)


In [100]:
train_2d = pd.read_csv('../processed/train_2d.csv')
test_2d = pd.read_csv('../processed/test_2d.csv')
test_full = pd.read_csv('../processed/test_full.csv')

In [101]:
train_2d.head()

,unit_number,time_cycles,setting_1,setting_2,T24,T30,T50,P15,P30,Nf,Nc,Ps30,phi,NRf,NRc,BPR,htBleed,W31,W32,RUL
0,1,1,0.459770,0.166667,0.183735,0.406802,0.309757,1.0,0.726248,0.242424,0.109755,0.369048,0.633262,0.205882,0.199608,0.363986,0.333333,0.713178,0.724662,125
1,1,2,0.609195,0.250000,0.283133,0.453019,0.352633,1.0,0.628019,0.212121,0.100242,0.380952,0.765458,0.279412,0.162813,0.411312,0.333333,0.666667,0.731014,125
2,1,3,0.252874,0.750000,0.343373,0.369523,0.370527,1.0,0.710145,0.272727,0.140043,0.250000,0.795309,0.220588,0.171793,0.357445,0.166667,0.627907,0.621375,125
3,1,4,0.540230,0.500000,0.343373,0.256159,0.331195,1.0,0.740741,0.318182,0.124518,0.166667,0.889126,0.294118,0.174889,0.166603,0.333333,0.573643,0.662386,125
4,1,5,0.390805,0.333333,0.349398,0.257467,0.404625,1.0,0.668277,0.242424,0.149960,0.255952,0.746269,0.235294,0.174734,0.402078,0.416667,0.589147,0.704502,125


In [98]:
test_2d.head()

,unit_number,time_cycles,setting_1,setting_2,T24,T30,T50,P15,P30,Nf,Nc,Ps30,phi,NRf,NRc,BPR,htBleed,W31,W32,RUL_True
0,1,31,0.465517,0.833333,0.412651,0.221932,0.281229,1.0,0.735910,0.272727,0.155569,0.226190,0.660981,0.264706,0.155692,0.298192,0.416667,0.519380,0.636564,112
1,2,49,0.603448,0.416667,0.403614,0.339002,0.482444,1.0,0.590982,0.303030,0.103383,0.488095,0.650320,0.308824,0.139127,0.483263,0.250000,0.519380,0.507595,98
2,3,126,0.408046,0.833333,0.503012,0.407892,0.618501,1.0,0.441224,0.393939,0.123530,0.613095,0.456290,0.382353,0.162659,0.334744,0.583333,0.612403,0.524441,69
3,4,106,0.568966,0.833333,0.472892,0.512099,0.415766,1.0,0.449275,0.348485,0.132684,0.476190,0.680171,0.338235,0.173909,0.532897,0.583333,0.341085,0.502486,82
4,5,98,0.425287,0.166667,0.319277,0.412034,0.626435,1.0,0.553945,0.303030,0.144755,0.363095,0.492537,0.397059,0.133141,0.428242,0.500000,0.472868,0.714582,91


In [99]:
test_full.head()

,unit_number,time_cycles,setting_1,setting_2,T24,T30,T50,P15,P30,Nf,Nc,Ps30,phi,NRf,NRc,BPR,htBleed,W31,W32
0,1,1,0.632184,0.750000,0.545181,0.310661,0.269413,1.0,0.652174,0.212121,0.127614,0.208333,0.646055,0.220588,0.132160,0.308965,0.333333,0.558140,0.661834
1,1,2,0.344828,0.250000,0.150602,0.379551,0.222316,1.0,0.805153,0.166667,0.146684,0.386905,0.739872,0.264706,0.204768,0.213159,0.416667,0.682171,0.686827
2,1,3,0.517241,0.583333,0.376506,0.346632,0.322248,1.0,0.685990,0.227273,0.158081,0.386905,0.699360,0.220588,0.155640,0.458638,0.416667,0.728682,0.721348
3,1,4,0.741379,0.500000,0.370482,0.285154,0.408001,1.0,0.679549,0.196970,0.105717,0.255952,0.573561,0.250000,0.170090,0.257022,0.250000,0.666667,0.662110
4,1,5,0.580460,0.500000,0.391566,0.352082,0.332039,1.0,0.694042,0.166667,0.102396,0.273810,0.737740,0.220588,0.152751,0.300885,0.166667,0.658915,0.716377
